<a href="https://colab.research.google.com/github/cbonnin88/Segment-IQ/blob/main/Customer_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install streamlit
!pip install pyngrok

In [53]:
%%writefile app.py
import streamlit as st
import polars as pl
import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from polars.datatypes.classes import Categories

# --- Page Configuration ---
st.set_page_config(page_title='SegmentIQ',layout='wide')
st.title('🛍️ SegmentIQ: Customer Insights Hub')
st.markdown('Filter demographics, segment customers, and predict future spending.')

# --- Data Loading ---
@st.cache_data
def load_data(file):
    try:
        if file is not None:
            pdf = pd.read_csv(file)
            return pl.read_csv(file)
        else:
            return pl.read_csv('customer_shopping_behavior.csv')
    except Exception as e:
        st.error(f'⚠️ Data Loading Error: {e}')
        return None

# Main application logic
st.sidebar.header('1. Data Source')
uploaded_file = st.sidebar.file_uploader('Upload your Customer CSV',type=['csv'])

df_customer = load_data(uploaded_file)

if df_customer is None:
    st.info('👋 Welcome! Please open the sidebar on the left and upload your CSV file to begin.')
else:
    try:
        df_customer_clean = df_customer.drop_nulls(subset=['Age','Purchase Amount','Gender','Category'])

        # --- Sidebar: Filters ---
        st.sidebar.header('2. Analyze Specific Demographics')
        genders = df_customer_clean['Gender'].unique().to_list()
        categories = df_customer_clean['Category'].unique().to_list()

        selected_genders = st.sidebar.multiselect('Select Genders:',options=genders,default=genders)
        selected_categories = st.sidebar.multiselect('Select Categories:',options=categories,default=categories)

        # Filtering the data
        df_filtered = df_customer_clean.filter(
            (pl.col('Gender').is_in(selected_genders)) &
            (pl.col('Category').is_in(selected_categories))
        )

        # --- Organizing the App into Tabs ---
        tab1, tab2 = st.tabs(["📊 Customer Segmentation", "🔮 Spend Predictor"])

        # --- Tab 1: Segmentation & Metrics & Export ---
        with tab1:
            if df_filtered.height < 2:
                st.error('Not enought data. Please adjust your filters')
            else:
                # Feature A: Metic Summaries
                st.subheader('High-Level Metrics')
                col1, col2,col3 = st.columns(3)

                avg_spend = df_filtered['Purchase Amount'].mean()
                av_age = df_filtered['Age'].mean()

                col1.metric('Total Customers',f'{df_filtered.height:,}')
                col2.metric('Average Spend',f'€{avg_spend:.2f}')
                col3.metric('Average Age',f'{av_age:.1f} years')
                st.divider()

                # --- Machine Learning: Clustering ---
                st.sidebar.header('3. Segmentation Settings')
                num_clusters = st.sidebar.slider('Number of Segments (k)', min_value=2, max_value=7,value=3)

                X = df_filtered.select(['Age','Purchase Amount']).to_numpy()
                scaler = StandardScaler()
                X_scaled = scaler.fit_transform(X)

                kmeans = KMeans(n_clusters=num_clusters, random_state=42,n_init='auto')
                kmeans.fit(X_scaled)

                # Add Segments to data and convert to Pandas for Plotly/Download
                df_segmented = df_filtered.with_columns(
                    pl.Series(name='Segment',values=kmeans.labels_.astype(str))
                )
                pdf_segmented = df_segmented.to_pandas()

                # Plotly Visualization
                fig = px.scatter(
                    pdf_segmented,
                    x='Age',
                    y='Purchase Amount',
                    color='Segment',
                    hover_data=['Gender','Category','Item Purchased'],
                    title='Customer Segment Map',
                    color_discrete_sequence=px.colors.sequential.Viridis[::-1],
                    opacity=0.8
                )
                fig.update_layout(plot_bgcolor='white')
                st.plotly_chart(fig, use_container_width=True)

                # --- Feature B: Data Export ---
                st.subheader('Export Segmented Data')
                st.markdown('Download this specific view of the data, complete with Machine Learning segment labels.')

                # Convert dataframe to CSV string
                csv_export = pdf_segmented.to_csv(index=False).encode('utf-8')

                st.download_button(
                    label='📥 Download CSV',
                    data=csv_export,
                    file_name ='segmented_customers.csv',
                    mime='text/csv'
                )

        # --- Tab 2: Predictive Machine Learning
        with tab2:
            st.subheader('Predict Customer Spending')
            st.markdown('We have trained a Random Forest AI on the filtered data. Enter a hypothetical customers details below to predict how much they will spend')

            # Training a predictive model
            df_ml = df_filtered.to_pandas()

            # Features X and Target y
            X_train = pd.get_dummies(df_ml[['Age','Gender','Category']])
            y_train = df_ml['Purchase Amount']

            # Train the Model
            regressor = RandomForestRegressor(n_estimators=100,random_state=42)
            regressor.fit(X_train, y_train)

            # Create users input to build a customer
            col_input1, col_input2, col_input3 = st.columns(3)
            with col_input1:
                input_age = st.number_input('Customer Age', min_value=18, max_value=90, value=30)
            with col_input2:
                input_gender = st.selectbox('Customer Gender',options=genders)

            with col_input3:
                input_category = st.selectbox('Product Category', options=categories)

            if st.button('🔮 Predict Spend'):
                # Create a dataframe for our single customer
                input_data = pd.DataFrame({'Age':[input_age],'Gender':[input_gender],'Category':[input_category]})

                # Processing the fake customer Exactly the same way
                input_encoded = pd.get_dummies(input_data)

                # Filling in missing columns with 0's
                input_encoded = input_encoded.reindex(columns=X_train.columns,fill_value=0)

                # Make the prediction
                prediction = regressor.predict(input_encoded)[0]

                st.success(f'### Predicted Purchase Amount: **€{prediction:.2f}**')
                st.info('💡 Note: This prediction adapts based on the active filters you have set in the sidebar!')

    except Exception as e:
        st.error(f'⚠️ Application Error: {e}')

Overwriting app.py


In [54]:
import subprocess
import time
from pyngrok import ngrok
from google.colab import userdata
import os

os.system('pkill -f streamlit')
ngrok.kill()

ngrok_id = userdata.get('ngrok_token')
ngrok.set_auth_token(ngrok_id)

# Then run the server and tunnel again:
import subprocess
import time

subprocess.Popen(["nohup", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(3)
public_url = ngrok.connect(8501)
print(f"🎉 SegmentIQ app is live! Click here: {public_url}")

🎉 SegmentIQ app is live! Click here: NgrokTunnel: "https://db7b-34-186-78-233.ngrok-free.app" -> "http://localhost:8501"


# **Creating a Customer CSV to test SegmentIQ**

In [55]:
import pandas as pd
import numpy as np
import random

In [56]:
# Generating fake customers
num_customers = 500

In [57]:
# Generate random data
data = {
    'Customer ID': range(1000,1000 + num_customers),
    'Age': np.random.randint(18,90,size=num_customers),
    'Gender': np.random.choice(['Male','Female','Non-Binary'], size=num_customers, p=[0.48,0.48,0.04]),
    "Category": np.random.choice(["Clothing", "Electronics", "Footwear", "Accessories"], size=num_customers),
    'Purchase Amount': np.random.uniform(20,500, size=num_customers).round(2)
}

In [58]:
df_new_customers = pd.DataFrame(data)

display(df_new_customers.head())

,Customer ID,Age,Gender,Category,Purchase Amount
0,1000,32,Male,Accessories,148.21
1,1001,86,Male,Accessories,277.11
2,1002,25,Female,Footwear,435.82
3,1003,47,Male,Clothing,491.67
4,1004,43,Female,Accessories,408.05


In [59]:
# Add Items Purchased based on the category
item_map = {
    "Clothing": ["T-shirt", "Jeans", "Jacket", "Sweater", "Dress"],
    "Electronics": ["Smartphone", "Headphones", "Laptop", "Smartwatch", "Tablet"],
    "Footwear": ["Running Shoes", "Sneakers", "Boots", "Sandals", "Heels"],
    "Accessories": ["Backpack", "Sunglasses", "Watch", "Belt", "Wallet"]
}

In [60]:
df_new_customers['Item Purchased'] = df_new_customers['Category'].apply(lambda x:random.choice(item_map[x]))

In [61]:
# Saving to a csv
file_name = 'test_customers.csv'
df_new_customers.to_csv(file_name, index=False)

print(f'✅ Successfully created {file_name} with {num_customers} rows!')

✅ Successfully created test_customers.csv with 500 rows!
